# TikTok Comment Sentiment with Scavio

Gauge how viewers react to a TikTok trend: find a top video for a keyword, pull its comments, and classify the audience sentiment. This recipe pairs the Scavio **Python SDK** (to fetch and trim the data) with an LLM (to read it) -- the right pattern when raw social payloads are too large to hand an agent directly.

**What you will learn:**
- Find videos with `client.tiktok.search_videos`
- Pull and trim comments with `client.tiktok.video_comments`
- Classify sentiment with a single LLM call over a compact comment list

In [1]:
# pip install scavio langchain-openai python-dotenv

In [2]:
from dotenv import load_dotenv
from scavio import ScavioClient
from langchain_openai import ChatOpenAI

load_dotenv(override=True)
client = ScavioClient()  # reads SCAVIO_API_KEY

In [3]:
KEYWORD = "cold brew coffee"

# 1. Find a popular recent video and grab its id (search payloads are huge,
#    so we extract only the few fields we need).
videos = client.tiktok.search_videos(keyword=KEYWORD, count=10).get("data", {})
items = videos.get("aweme_list") or videos.get("search_item_list") or []

def field(it, key):
    return it.get(key) or (it.get("aweme_info") or {}).get(key)

items = [it for it in items if field(it, "aweme_id")]
items.sort(key=lambda it: (field(it, "statistics") or {}).get("play_count", 0), reverse=True)
top = items[0]
video_id = field(top, "aweme_id")
print("Top video:", video_id)
print("Caption:", (field(top, "desc") or "")[:120])

Top video: 7592042098237394206
Caption: ☕️ Tasting and ranking cold brew coffee! Yes I was highly caffeinated after filming this 😵‍💫 No, none of the coffee went


In [4]:
# 2. Pull comments and keep only text + like count.
raw = client.tiktok.video_comments(video_id=video_id, count=50).get("data", {})
comments = [
    {"text": c.get("text", "").strip(), "likes": c.get("digg_count", 0)}
    for c in (raw.get("comments") or [])
    if c.get("text")
]
comments.sort(key=lambda c: c["likes"], reverse=True)
comments = comments[:40]
print(f"Collected {len(comments)} comments")
for c in comments[:5]:
    print(f"  ({c['likes']:>4} likes) {c['text'][:80]}")

Collected 40 comments
  (6403 likes) How jittery were you after this 😂
  (5703 likes) [Sticker] When she finished filming…
  (5371 likes) Cafe Bustelo will have you cleaning and reorganizing your entire house in one wh
  (4133 likes) Ok I buy Stok religiously because they display caffeine content on the bottle wh
  (2088 likes) I only drink Stok so I’m feeling validated lmao


In [5]:
# 3. One LLM call over the compact list -> sentiment summary.
comment_block = "\n".join(f"- ({c['likes']} likes) {c['text']}" for c in comments)
prompt = (
    f'These are top comments on a TikTok video about "{KEYWORD}".\n\n'
    f"{comment_block}\n\n"
    "Report: 1) overall sentiment (positive/mixed/negative), 2) the top 3 themes, "
    "3) two representative quotes. Base everything only on these comments."
)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print(llm.invoke(prompt).content)

1) **Overall Sentiment**: Positive

2) **Top 3 Themes**:
   - **Brand Preference**: Many comments express a strong preference for specific cold brew brands, particularly Stok and Cafe Bustelo.
   - **Caffeine Effects**: Several comments discuss the effects of caffeine from cold brew, including feelings of jitteriness and energy.
   - **Taste and Quality**: Users share their opinions on the taste and quality of different cold brew brands, often comparing them to one another.

3) **Two Representative Quotes**:
   - "I only drink Stok so I’m feeling validated lmao"
   - "Cafe Bustelo will have you cleaning and reorganizing your entire house in one whole day. That stuff is serious 😂"
